# 04 · Results and the choice of the final model

1. Scores of every run, as mean ± SD over the 5 outer test folds, on **all test cells** (measured + filled, the main result) and on **measured cells only**.
2. The final model is chosen by the rule decided **before** the results (same as V3):
   * eligible: mean Clay and Sand recall ≥ 0.50, and no fold below 0.25;
   * best = highest mean Macro-F1;
   * among the eligible models within one standard error of the best: lowest fold SD of Macro-F1, then lowest Brier score.
3. Scores by distance to the nearest training borehole, block-size sensitivity and purge.

**Run time:** under 1 minute.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])
preds = pd.read_csv(H.out_dir(S, "03_spatial_cv", private=True) / "outer_predictions.csv")
OUT = H.out_dir(S, "04_results")
folds_tab = H.fold_scores(preds)
summ = H.summary(folds_tab)
folds_tab.to_csv(OUT / "scores_by_fold.csv", index=False)
summ.to_csv(OUT / "scores_mean_sd.csv", index=False)
show = ["model", "macro_f1_mean", "macro_f1_sd", "balanced_accuracy_mean", "recall_clay_mean", "recall_sand_mean",
        "roc_auc_mean", "brier_mean", "n_folds"]
main = summ[(summ.multiple == 1) & (summ.purge_x == 0)]
for c in ("all", "measured"):
    print(f"Main block size ({main.block_m.iloc[0]:.0f} m), {c} test cells:")
    display(main[main.cells == c][show].sort_values("macro_f1_mean", ascending=False).round(3))

## 1. The final model (rule decided before the results)

In [ ]:
sel = H.select_model(summ, "all")
display(sel["table"][["model", "eligible", "macro_f1_mean", "macro_f1_sd", "brier_mean"]].round(3))
print("Highest mean Macro-F1:", sel.get("highest_mean"), "| one SE:", round(sel.get("one_se", np.nan), 4))
print("Within one SE:", sel.get("within_1se"))
print("FINAL MODEL:", sel["selected"])
sel_m = H.select_model(summ, "measured")
print("(for comparison, the same rule on measured cells only would choose:", sel_m["selected"], ")")
_ = (OUT / "selection.json").write_text(json.dumps({k: v for k, v in sel.items() if k != "table"}, indent=1, default=str))

## 2. Model comparison figure

In [ ]:
m = main[main.cells == "all"].sort_values("macro_f1_mean", ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#9E9E9E" if x == H.DEPTH_ONLY else ("#A0004E" if x == sel["selected"] else "#17365C") for x in m.model]
ax.bar(m.model, m.macro_f1_mean, yerr=m.macro_f1_sd, color=colors, capsize=4)
for i, v in enumerate(m.macro_f1_mean):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
ax.set_ylabel("Macro-F1 (mean ± SD, 5 outer folds)"); ax.set_ylim(0, 1)
ax.set_title(f"Spatial CV, {m.block_m.iloc[0]:.0f} m blocks (magenta = final model, grey = depth-only reference)", fontsize=9)
fig.savefig(OUT / "fig_model_comparison.png", dpi=200, bbox_inches="tight")

## 3. By distance to the nearest training borehole
Does a model beat the depth-only reference close to boreholes, far from them, or both?

In [ ]:
bands = H.distance_bands(preds[(preds.multiple == 1) & (preds.purge_x == 0)])
bands.to_csv(OUT / "scores_by_distance.csv", index=False)
t = bands[bands.cells == "all"].pivot_table(index="model", columns="band", values="macro_f1", observed=True)
n = bands[bands.cells == "all"].groupby("band", observed=True).n.max()
print("Macro-F1 by distance band (all cells); test cells per band:", n.to_dict())
display(t.round(3))
fig, ax = plt.subplots(figsize=(7, 4))
for model, row in t.iterrows():
    ax.plot(row.index.astype(str), row.values, "-o", label=model,
            color="#9E9E9E" if model == H.DEPTH_ONLY else None, linewidth=2.5 if model == sel["selected"] else 1.2)
ax.set_ylabel("Macro-F1"); ax.set_xlabel("Distance to the nearest training borehole"); ax.legend(frameon=False, fontsize=8)
fig.savefig(OUT / "fig_scores_by_distance.png", dpi=200, bbox_inches="tight")

## 4. Block-size sensitivity and purge

In [ ]:
sens = summ[(summ.purge_x == 0) & (summ.cells == "all") & summ.model.isin(S["cv"]["sensitivity_models"] + [H.DEPTH_ONLY])]
display(sens.pivot_table(index="model", columns="block_m", values="macro_f1_mean").round(3))
pur = summ[(summ.multiple == 1) & (summ.cells == "all") & summ.model.isin(S["cv"]["purge_models"])]
display(pur.pivot_table(index="model", columns="purge_m", values="macro_f1_mean").round(3))
print(f"Finished in {time.time() - T0:.0f} s. Next: notebook 05.")